# Chapter 22 — Can a Smaller Representation Preserve a Larger One?

**Book alignment:** Embeddings From First Principles, Chapter 22

**Question this notebook isolates:** Is whole-document embedding drift a *faithfulness*
check? On RELATE-DOC (Wave 4), across eight controlled corruptions — a dropped number, a
reversed relation, a negated claim — does layer-1 global drift detect **0%**, does claim-
conditioned similarity (layer 4) catch only the *deletions*, and does only an external NLI
verifier (layer 5) catch a reversed fact?

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave, name):
    return json.loads((EXP / wave / "artifacts" / f"{name}.json").read_text())

## 1. The five-layer checksum stack, and its measured blind spot (Wave 4)

In [ ]:
bs = art("wave4", "blindspot-matrix")
band = bs["calibration"]["band"]
print(f"calibration: faithful-vs-own-doc {bs['calibration']['faithful_vs_own_doc_mean']:.2f}, "
      f"faithful-vs-other-doc {bs['calibration']['faithful_vs_other_doc_mean']:.2f}, band {band}")
print()
layers = ["L1_global_drift", "L2_neighborhood", "L3_query_conditioned",
          "L4_claim_conditioned", "L5_nli_verification"]
print(f"{'corruption':24} " + "  ".join(l.split('_')[0] for l in layers))
for corr, row in bs["matrix"].items():
    print(f"{corr:24} " + "  ".join(f"{row[l]:>4.2f}" for l in layers))

In [ ]:
m = bs["matrix"]
# layer 1 (whole-document drift) detects NOTHING - not a reversal, not a negation, not a number
assert all(m[c]["L1_global_drift"] == 0.0 for c in m)
# claim-conditioned (L4) catches the DELETIONS ...
assert m["minority_entity_dropped"]["L4_claim_conditioned"] >= 0.5
assert m["number_dropped"]["L4_claim_conditioned"] >= 0.3
# ... but is near-blind to reversals and value swaps
assert m["negation_inserted"]["L4_claim_conditioned"] < 0.1
assert m["relation_reversed"]["L4_claim_conditioned"] == 0.0
assert m["number_changed"]["L4_claim_conditioned"] == 0.0
# only the external NLI verifier catches every corruption, with 0 false positives on 'faithful'
assert min(m[c]["L5_nli_verification"] for c in m if c != "faithful") >= 0.85
assert m["faithful"]["L5_nli_verification"] == 0.0
print("coarse geometric preservation is systematically compatible with fine semantic failure")
print("a compression can hold a document's location to 3 decimals while reversing the one fact that matters")

## What we earned

The "semantic checksum" is a layered stack — global drift → neighbourhood → query-conditioned
→ claim-conditioned → external NLI — each layer catching what the previous misses. On
RELATE-DOC, layer-1 whole-document drift detected **0%** of every corruption class; claim-
conditioned similarity caught the *deletions* (and, partly, a changed conclusion); a
reversed relation or a changed number slipped past every embedding layer and needed an
external verifier. Global geometric preservation gates *topical* drift, not faithfulness.

**Notebook 23 / Chapter 23** looks at the difference between two vectors as an object —
`E(x₂) − E(x₁)` — and asks whether a semantic edit is a reusable operator.